# Exercise 03 : An Agent with Tests

- `create_agent` compiles a tool-calling loop
- A scripted fake model runs that loop without the network
- `ModelCallLimitMiddleware` stops the loop after a fixed number of model calls

You will look up a part in Aurelia's `service.db`, test the agent with a fake
model, and make one live call. The Aurelia records are made up.

Worked examples to revisit : module 04, demo02 · create-agent, demo05 · agent-failures,
demo07 · testing-agents.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [ ]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

## Step 1 : Tool

Read `stock` for one part number. Return `Unknown part` when the row is missing.

In [ ]:
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = Path(data("capstone/service.db")).resolve()


@tool
def part_stock(part_no: str) -> str:
    """Read recorded units in stock for an Aurelia spare part."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
        row = conn.execute(____).fetchone()
    return str(row[0]) if row else "Unknown part"

<details><summary>Hint</summary>

The slide *Defining Tools* shows parameterised SQL with `conn.execute`, and module 04 demo07 · testing-agents reads one row the same way. The blank takes the query string and a tuple with its parameter. Select the `stock` column of the `parts` table for the tool's argument.

</details>

In [ ]:
known = part_stock.invoke({"part_no": "CT-SR-03"})
unknown = part_stock.invoke({"part_no": "UNKNOWN"})
print("CT-SR-03 :", known)
print("UNKNOWN :", unknown)
print("check :", "pass" if known == "0" and unknown == "Unknown part" else "fail")

## Step 2 : Scripted model

`GenericFakeChatModel.bind_tools` raises `NotImplementedError`. Return the
same model from the subclass, then script one tool call and a final answer.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolCall
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel


class FakeToolModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return ____


def call(part_no="CT-SR-03", call_id="stock-1"):
    return AIMessage("", tool_calls=[ToolCall(name="part_stock", args={"part_no": part_no}, id=call_id)])


agent = create_agent(
    FakeToolModel(messages=iter([call(), AIMessage("CT-SR-03 has no stock.")])),
    tools=[part_stock],
    system_prompt="Use part_stock for stock questions.")
out = agent.invoke({"messages": [HumanMessage("Stock of CT-SR-03?")]})
for message in out["messages"]:
    print(message.type, message.content or getattr(message, "tool_calls", []),
          getattr(message, "tool_call_id", ""))

<details><summary>Hint</summary>

The slide *Unit Tests with a Fake Model* and module 04 demo07 · testing-agents show the `bind_tools` override. `create_agent` uses whatever `bind_tools` returns, and the step says which model that is.

</details>

In [ ]:
results = [m for m in out["messages"] if m.type == "tool"]
ok = len(results) == 1 and results[0].content == "0" and results[0].tool_call_id == "stock-1"
print("check :", "pass" if ok else "fail")

## Step 3 : Pytest

The test module repeats the tool checks and stops a second model call with
`ModelCallLimitMiddleware`. Pytest uses the fake model only.

In [ ]:
from pathlib import Path
import subprocess
import sys

TEST = r"""
import sqlite3
from pathlib import Path

import pytest
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.messages import AIMessage, HumanMessage, ToolCall
from langchain.tools import tool
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel


def service_db() -> Path:
    here = Path(__file__).resolve().parent
    root = next(p for p in [here, *here.parents] if (p / "data" / "capstone" / "service.db").exists())
    return root / "data" / "capstone" / "service.db"


@tool
def part_stock(part_no: str) -> str:
    "Read recorded units in stock for an Aurelia spare part."
    with sqlite3.connect(f"file:{service_db().as_posix()}?mode=ro", uri=True) as conn:
        row = conn.execute("SELECT stock FROM parts WHERE part_no = ?", (part_no,)).fetchone()
    return str(row[0]) if row else "Unknown part"


class FakeToolModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


def call(part_no="CT-SR-03", call_id="stock-1"):
    return AIMessage("", tool_calls=[ToolCall(name="part_stock", args={"part_no": part_no}, id=call_id)])


def scripted(*messages, middleware=()):
    return create_agent(
        FakeToolModel(messages=iter(messages)),
        tools=[part_stock],
        middleware=list(middleware),
        system_prompt="Use part_stock for stock questions.",
    )


def test_known_and_unknown_part():
    assert part_stock.invoke({"part_no": "CT-SR-03"}) == "0"
    assert part_stock.invoke({"part_no": "UNKNOWN"}) == "Unknown part"


def test_tool_result_and_call_id():
    agent = scripted(call(), AIMessage("CT-SR-03 has no stock."))
    out = agent.invoke({"messages": [HumanMessage("Stock of CT-SR-03?")]})
    results = [m for m in out["messages"] if m.type == "tool"]
    assert len(results) == 1
    assert results[0].content == "0"
    assert results[0].tool_call_id == "stock-1"


def test_run_limit_stops_the_second_call():
    agent = scripted(
        call(call_id="c1"),
        call(call_id="c2"),
        AIMessage("done"),
        middleware=[____],
    )
    out = agent.invoke({"messages": [HumanMessage("Stock?")]})
    results = [m for m in out["messages"] if m.type == "tool"]
    assert len(results) == 1
    assert "run limit (1/1)" in out["messages"][-1].content
"""

path = Path("test_exercise03_agent.py")
path.write_text(TEST, encoding="utf-8")
print("wrote", path.name)

<details><summary>Hint</summary>

The slide *Model Call Limit* and module 04 demo05c · call-limits show this middleware. The test scripts two tool calls and expects one tool result and the text `run limit (1/1)`. Choose the limit and the exit behaviour that give that result.

</details>

In [ ]:
import shutil
import subprocess
import sys

try:
    run = subprocess.run(
        [sys.executable, "-B", "-m", "pytest", "-p", "no:cacheprovider", "-q", str(path)],
        capture_output=True, text=True)
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    print("check :", "pass" if run.returncode == 0 else "fail")
finally:
    path.unlink(missing_ok=True)
    shutil.rmtree(path.parent / "__pycache__", ignore_errors=True)

## Step 4 : Live run

The cell runs the same tool with a call limit of 4 on the course model and asks
for `CT-SR-03`.

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

live = create_agent(
    llm,
    tools=[part_stock],
    middleware=[ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")],
    system_prompt="Use part_stock for stock questions. Report the number the tool returns.")
live_out = live.invoke(
    {"messages": [HumanMessage("How many CT-SR-03 units are in stock? Use the tool.")]},
    {"recursion_limit": 8})
for message in live_out["messages"]:
    print(message.type, message.content or getattr(message, "tool_calls", []))
    if getattr(message, "usage_metadata", None):
        print("usage :", message.usage_metadata)

In [ ]:
tool_msgs = [m for m in live_out["messages"] if m.type == "tool"]
print("tool results :", [m.content for m in tool_msgs])
print("check :", "pass" if any(m.content == "0" for m in tool_msgs) else "fail")

## If time permits

- Add `ToolCallLimitMiddleware` on `part_stock` and script a second call
- Give the live agent a checkpointer, run two thread ids and compare `get_state`
- Change `run_limit` to 1 on the live agent and read the limit message